In [104]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [105]:
#Load data
bank = pd.read_csv('bank-additional-full.csv', sep=";")
cost_assumptions = pd.read_excel('campaign_cost_assumptions.xlsx')

In [106]:
print("Bank shape: ", bank.shape) 
print("Cost assumtion shape: ", cost_assumptions.shape) 

Bank shape:  (41188, 21)
Cost assumtion shape:  (4, 5)


In [107]:
display(bank.head())
display(cost_assumptions)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


,channel,campaign_version,planned_cost_per_customer,actual_cost_per_customer,estimated_contribution_per_conversion
0,cellular,control,18,20,300
1,cellular,test,22,25,300
2,telephone,control,30,35,300
3,telephone,test,38,45,300


In [108]:
# Clean column names
bank.columns = (
    bank.columns.str.lower().str.strip().str.replace(".","_", regex=False).str.replace(" ", "_", regex=False)
)

In [109]:
bank.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [110]:
# Remove duplicate rows
duplicate_count = bank.duplicated().sum()
print(duplicate_count)
bank = bank.drop_duplicates().copy()

12


In [111]:
bank.duplicated().sum()

np.int64(0)

In [112]:
# Create customer_id because UCI does not provide one
bank.insert(0, "customer_id", range(1, len(bank)+1))

In [113]:
bank.head()

,customer_id,age,job,marital,education,default,housing,loan,contact,month,...,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,1,56,housemaid,married,basic.4y,no,no,no,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,2,57,services,married,high.school,unknown,no,no,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,3,37,services,married,high.school,no,yes,no,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,4,40,admin.,married,basic.6y,no,no,no,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,5,56,services,married,high.school,no,no,yes,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [114]:
# Standardize text columns
text_cols = bank.select_dtypes(include='object').columns

for col in text_cols:
    bank[col] = bank[col].str.strip().str.lower()

In [115]:
# Convert target into numeric conversion flag
bank["converted"] = bank['y'].map({'yes': 1, 'no': 0})

In [116]:
# Create customer segments
bank['age_group'] = pd.cut(
    bank["age"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "55+"]
)

In [117]:
bank["campaign_pressure"] = pd.cut(
    bank["campaign"],
    bins=[0, 1, 3, 5, 100],
    labels= ["Low", "Medium", "High", "Very High"]
)

In [118]:
bank["previously_contacted"] = np.where(bank["previous"] > 0, "Yes", "No")

In [119]:
month_order = {
    "jan":1, "feb":2, "mar":3, "apr":4,
    "may":5, "jun":6, "jul":7, "aug":8,
    "sep":9, "oct":10, "nov":11, "dec":12
}

bank["month_num"] = bank["month"].map(month_order)
bank["month_name"] = bank["month"].str.title()

In [120]:
bank.head(5)

,customer_id,age,job,marital,education,default,housing,loan,contact,month,...,cons_conf_idx,euribor3m,nr_employed,y,converted,age_group,campaign_pressure,previously_contacted,month_num,month_name
0,1,56,housemaid,married,basic.4y,no,no,no,telephone,may,...,-36.4,4.857,5191.0,no,0,55+,Low,No,5,May
1,2,57,services,married,high.school,unknown,no,no,telephone,may,...,-36.4,4.857,5191.0,no,0,55+,Low,No,5,May
2,3,37,services,married,high.school,no,yes,no,telephone,may,...,-36.4,4.857,5191.0,no,0,36-45,Low,No,5,May
3,4,40,admin.,married,basic.6y,no,no,no,telephone,may,...,-36.4,4.857,5191.0,no,0,36-45,Low,No,5,May
4,5,56,services,married,high.school,no,no,yes,telephone,may,...,-36.4,4.857,5191.0,no,0,55+,Low,No,5,May


In [121]:
bank.columns

Index(['customer_id', 'age', 'job', 'marital', 'education', 'default',
       'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration',
       'campaign', 'pdays', 'previous', 'poutcome', 'emp_var_rate',
       'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'y',
       'converted', 'age_group', 'campaign_pressure', 'previously_contacted',
       'month_num', 'month_name'],
      dtype='object')

In [122]:
# Data quality flag
important_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "poutcome"]
bank["has_unknown_value"] = bank[important_cols].eq("unknown").any(axis=1)

quality_report = pd.DataFrame({
    "metric": [
        "duplicate_rows_removed",
        "final_rows",
        "rows_with_unknown_values",
        "overall_conversion_rate"
    ],
    "value": [
        duplicate_count,
        len(bank),
        bank["has_unknown_value"].sum(),
        bank["converted"].mean()
    ]
})

In [123]:
quality_report.to_csv("data_quality_report.csv", index=False)
bank.to_csv("clean_bank_data.csv", index=False)
display(quality_report)

,metric,value
0,duplicate_rows_removed,12.000000
1,final_rows,41176.000000
2,rows_with_unknown_values,10698.000000
3,overall_conversion_rate,0.112663


In [124]:
# Create the A/B experiment table
experiment = bank[bank["contact"].isin(["cellular", "telephone"])].copy()

In [125]:
# Function to assign control/test inside each channel-month group
rng = np.random.default_rng(42)

def assign_ab_group(group):
    n = len(group)
    labels = np.array(["control"] * (n//2) + ["test"] * (n - n // 2))
    rng.shuffle(labels)
    group = group.copy()
    group["campaign_version"] = labels
    return group

experiment = (
    experiment.groupby(["contact", "month"], group_keys=False)
    .apply(assign_ab_group)
)

In [126]:
# Rename contact to channel
experiment = experiment.rename(columns={"contact": "channel"})

In [127]:
# Create campaign_id: channel + version + month
experiment["campaign_id"] = (
    experiment["channel"].str[:3].str.upper()
    + "_"
    + experiment["campaign_version"].str.upper()
    + "_"
    + experiment["month_name"].str.upper()
)

In [128]:
# Create experiment fact table
fact_experiment = experiment[[
    "customer_id",
    "campaign_id",
    "month",
    "month_name",
    "month_num",
    "converted",
    "campaign",
    "previous",
    "poutcome",
    "campaign_pressure",
    "previously_contacted"
]].copy()

In [129]:
fact_experiment.insert(0, "experiment_id", range(1, len(fact_experiment) + 1))

In [130]:
fact_experiment.to_csv("fact_experiment.csv", index=False)
display(fact_experiment.head())

,experiment_id,customer_id,campaign_id,month,month_name,month_num,converted,campaign,previous,poutcome,campaign_pressure,previously_contacted
0,1,1,TEL_CONTROL_MAY,may,May,5,0,1,0,nonexistent,Low,No
1,2,2,TEL_TEST_MAY,may,May,5,0,1,0,nonexistent,Low,No
2,3,3,TEL_CONTROL_MAY,may,May,5,0,1,0,nonexistent,Low,No
3,4,4,TEL_CONTROL_MAY,may,May,5,0,1,0,nonexistent,Low,No
4,5,5,TEL_TEST_MAY,may,May,5,0,1,0,nonexistent,Low,No


In [131]:
# Create dimension tables
dim_customer = bank[[
    "customer_id",
    "age",
    "age_group",
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "has_unknown_value"
]].drop_duplicates()

dim_customer.to_csv("dim_customer.csv", index=False)

display(dim_customer.head())

,customer_id,age,age_group,job,marital,education,default,housing,loan,has_unknown_value
0,1,56,55+,housemaid,married,basic.4y,no,no,no,False
1,2,57,55+,services,married,high.school,unknown,no,no,True
2,3,37,36-45,services,married,high.school,no,yes,no,False
3,4,40,36-45,admin.,married,basic.6y,no,no,no,False
4,5,56,55+,services,married,high.school,no,no,yes,False


In [132]:
# Create dim_campaign
dim_campaign = experiment[[
    "campaign_id",
    "channel",
    "campaign_version",
    "month",
    "month_name",
    "month_num"
]].drop_duplicates()

dim_campaign.to_csv("dim_campaign.csv", index=False)

display(dim_campaign.head())

,campaign_id,channel,campaign_version,month,month_name,month_num
0,TEL_CONTROL_MAY,telephone,control,may,May,5
1,TEL_TEST_MAY,telephone,test,may,May,5
7763,TEL_CONTROL_JUN,telephone,control,jun,Jun,6
7765,TEL_TEST_JUN,telephone,test,jun,Jun,6
12137,TEL_CONTROL_JUL,telephone,control,jul,Jul,7


In [133]:
# Create the campaign cost fact table

# Clean cost assumption columns
cost_assumptions.columns = (
    cost_assumptions.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_", regex=False)
)

In [134]:
# Clean text columns
cost_assumptions["channel"] = (
    cost_assumptions["channel"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [135]:
cost_assumptions["campaign_version"] = (
    cost_assumptions["campaign_version"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [136]:
# If old column name exists, rename it to the new realistic business name
if "estimated_revenue_per_conversion" in cost_assumptions.columns:
    cost_assumptions = cost_assumptions.rename(
        columns={
            "estimated_revenue_per_conversion": "estimated_contribution_per_conversion"
        }
    )

In [137]:
# Required columns for this project
required_cols = [
    "channel",
    "campaign_version",
    "planned_cost_per_customer",
    "actual_cost_per_customer",
    "estimated_contribution_per_conversion"
]

In [138]:
missing_cols = [
    col for col in required_cols
    if col not in cost_assumptions.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns in campaign_cost_assumptions.xlsx: {missing_cols}. "
        f"Current columns are: {cost_assumptions.columns.tolist()}"
    )

In [139]:
# Convert numeric columns
numeric_cols = [
    "planned_cost_per_customer",
    "actual_cost_per_customer",
    "estimated_contribution_per_conversion"
]

for col in numeric_cols:
    cost_assumptions[col] = pd.to_numeric(
        cost_assumptions[col],
        errors="coerce"
    )

In [140]:
display(cost_assumptions)
print(cost_assumptions.dtypes)

,channel,campaign_version,planned_cost_per_customer,actual_cost_per_customer,estimated_contribution_per_conversion
0,cellular,control,18,20,300
1,cellular,test,22,25,300
2,telephone,control,30,35,300
3,telephone,test,38,45,300


channel                                  object
campaign_version                         object
planned_cost_per_customer                 int64
actual_cost_per_customer                  int64
estimated_contribution_per_conversion     int64
dtype: object


In [141]:
# Count customers and conversions by campaign_id
campaign_volume = (
    experiment
    .groupby(["campaign_id", "channel", "campaign_version", "month_name", "month_num"])
    .agg(
        customers_contacted=("customer_id", "count"),
        conversions=("converted", "sum")
    )
    .reset_index()
)

In [142]:
# Join with cost assumptions
fact_campaign_cost = campaign_volume.merge(
    cost_assumptions,
    on=["channel", "campaign_version"],
    how="left"
)

In [145]:
# Calculate finance metrics
fact_campaign_cost["planned_budget"] = (
    fact_campaign_cost["customers_contacted"] *
    fact_campaign_cost["planned_cost_per_customer"]
)

fact_campaign_cost["actual_spend"] = (
    fact_campaign_cost["customers_contacted"] *
    fact_campaign_cost["actual_cost_per_customer"]
)

fact_campaign_cost["estimated_contribution"] = (
    fact_campaign_cost["conversions"] *
    fact_campaign_cost["estimated_contribution_per_conversion"]
)

fact_campaign_cost["estimated_profit"] = (
    fact_campaign_cost["estimated_contribution"] -
    fact_campaign_cost["actual_spend"]
)

fact_campaign_cost["budget_variance"] = (
    fact_campaign_cost["planned_budget"] -
    fact_campaign_cost["actual_spend"]
)

fact_campaign_cost["budget_variance_pct"] = (
    fact_campaign_cost["budget_variance"] /
    fact_campaign_cost["planned_budget"]
)

fact_campaign_cost["budget_status"] = np.where(
    fact_campaign_cost["actual_spend"] > fact_campaign_cost["planned_budget"],
    "Over Budget",
    "Within Budget"
)

In [146]:
fact_campaign_cost.to_csv("fact_campaign_cost.csv", index=False)

display(fact_campaign_cost.head())

display(fact_campaign_cost)

,campaign_id,channel,campaign_version,month_name,month_num,customers_contacted,conversions,planned_cost_per_customer,actual_cost_per_customer,estimated_contribution_per_conversion,planned_budget,actual_spend,estimated_contribution,estimated_profit,budget_variance,budget_variance_pct,budget_status
0,CEL_CONTROL_APR,cellular,control,Apr,4,1222,241,18,20,300,21996,24440,72300,47860,-2444,-0.111111,Over Budget
1,CEL_CONTROL_AUG,cellular,control,Aug,8,2953,318,18,20,300,53154,59060,95400,36340,-5906,-0.111111,Over Budget
2,CEL_CONTROL_DEC,cellular,control,Dec,12,74,40,18,20,300,1332,1480,12000,10520,-148,-0.111111,Over Budget
3,CEL_CONTROL_JUL,cellular,control,Jul,7,3046,289,18,20,300,54828,60920,86700,25780,-6092,-0.111111,Over Budget
4,CEL_CONTROL_JUN,cellular,control,Jun,6,410,168,18,20,300,7380,8200,50400,42200,-820,-0.111111,Over Budget


,campaign_id,channel,campaign_version,month_name,month_num,customers_contacted,conversions,planned_cost_per_customer,actual_cost_per_customer,estimated_contribution_per_conversion,planned_budget,actual_spend,estimated_contribution,estimated_profit,budget_variance,budget_variance_pct,budget_status
0,CEL_CONTROL_APR,cellular,control,Apr,4,1222,241,18,20,300,21996,24440,72300,47860,-2444,-0.111111,Over Budget
1,CEL_CONTROL_AUG,cellular,control,Aug,8,2953,318,18,20,300,53154,59060,95400,36340,-5906,-0.111111,Over Budget
2,CEL_CONTROL_DEC,cellular,control,Dec,12,74,40,18,20,300,1332,1480,12000,10520,-148,-0.111111,Over Budget
3,CEL_CONTROL_JUL,cellular,control,Jul,7,3046,289,18,20,300,54828,60920,86700,25780,-6092,-0.111111,Over Budget
4,CEL_CONTROL_JUN,cellular,control,Jun,6,410,168,18,20,300,7380,8200,50400,42200,-820,-0.111111,Over Budget
5,CEL_CONTROL_MAR,cellular,control,Mar,3,243,132,18,20,300,4374,4860,39600,34740,-486,-0.111111,Over Budget
6,CEL_CONTROL_MAY,cellular,control,May,5,2758,324,18,20,300,49644,55160,97200,42040,-5516,-0.111111,Over Budget
7,CEL_CONTROL_NOV,cellular,control,Nov,11,1837,180,18,20,300,33066,36740,54000,17260,-3674,-0.111111,Over Budget
8,CEL_CONTROL_OCT,cellular,control,Oct,10,281,129,18,20,300,5058,5620,38700,33080,-562,-0.111111,Over Budget
9,CEL_CONTROL_SEP,cellular,control,Sep,9,241,126,18,20,300,4338,4820,37800,32980,-482,-0.111111,Over Budget
